# Inspect BM25: Sparse Embedding Pipeline

Before running the full BM25 pipeline (`build_bm25.py` to train, then `apply_bm25.py` to embed), this notebook demonstrates the BM25 sparse embedding approach on a small sample of chunks — one from each source type (Reddit, Zoom, PDF). It shows:

1. **Preprocessing** — How `chunk_prep()` transforms raw text: lowercasing, punctuation removal, lemmatization, stopword removal, n-gram generation
2. **BM25 input construction** — How Approach B concatenates `text_content` with enrichment tags to normalize concepts across source types
3. **Model training** — `BM25Okapi` on the sample corpus, inspecting IDF, vocabulary size, and average document length
4. **Sparse embedding** — Converting BM25 scores to sparse vectors (indices + values) for Vector Search 2.0
5. **Query embedding** — How a search query gets the same sparse treatment, and how it overlaps with chunk embeddings

---
## Setup

In [ ]:
import string
from collections import Counter

import nltk
import pandas as pd
from rank_bm25 import BM25Okapi
from google.cloud import bigquery
from config import (
    BQ_PROJECT, BQ_DATASET, BQ_TABLE_PREFIX,
    BM25_K1, BM25_B, BM25_EPSILON, BM25_NGRAMS,
)

nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

lemmatizer = nltk.stem.WordNetLemmatizer()
stop_words = set(nltk.corpus.stopwords.words('english'))

bq = bigquery.Client(project=BQ_PROJECT)

print(f'BM25 params: k1={BM25_K1}, b={BM25_B}, epsilon={BM25_EPSILON}, ngrams={BM25_NGRAMS}')

---
## Read Sample Chunks

Pull one chunk from each type, including enrichment tag columns, to demonstrate the full BM25 input construction.

In [ ]:
reddit_row = list(bq.query(
    f"SELECT chunk_id, text_content, topics, methods_mentioned "
    f"FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_reddit_chunks` LIMIT 1"
).result())[0]

zoom_row = list(bq.query(
    f"SELECT chunk_id, text_content, topics, action_items "
    f"FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_zoom_chunks` LIMIT 1"
).result())[0]

pdf_row = list(bq.query(
    f"SELECT chunk_id, text_content, topics, functions_referenced "
    f"FROM `{BQ_PROJECT}.{BQ_DATASET}.{BQ_TABLE_PREFIX}_pdf_chunks` LIMIT 1"
).result())[0]

print(f'Reddit: {reddit_row.chunk_id} ({len(reddit_row.text_content)} chars)')
print(f'  topics: {list(reddit_row.topics) if reddit_row.topics else []}')
print(f'  methods: {list(reddit_row.methods_mentioned) if reddit_row.methods_mentioned else []}')
print(f'Zoom:   {zoom_row.chunk_id} ({len(zoom_row.text_content)} chars)')
print(f'  topics: {list(zoom_row.topics) if zoom_row.topics else []}')
print(f'  action_items: {list(zoom_row.action_items) if zoom_row.action_items else []}')
print(f'PDF:    {pdf_row.chunk_id} ({len(pdf_row.text_content)} chars)')
print(f'  topics: {list(pdf_row.topics) if pdf_row.topics else []}')
print(f'  functions: {list(pdf_row.functions_referenced) if pdf_row.functions_referenced else []}')

---
## Preprocessing

`chunk_prep()` transforms raw text into BM25-ready tokens:
1. Lowercase the text
2. Remove punctuation
3. Tokenize into words
4. Lemmatize each token (reduce to root form: "forecasting" -> "forecasting", "models" -> "model")
5. Remove stopwords and single-character tokens
6. Generate n-grams (unigrams + bigrams by default)

In [ ]:
def chunk_prep(text, normalize='lemmatize', ngrams=BM25_NGRAMS):
    """Preprocess text for BM25: lowercase, remove punctuation, tokenize,
    lemmatize, remove stopwords, generate n-grams."""
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    tokens = nltk.tokenize.word_tokenize(text)
    unigrams = [lemmatizer.lemmatize(t) for t in tokens
                if len(t) > 1 and t not in stop_words]
    if ngrams >= 2:
        all_grams = unigrams.copy()
        for n in range(2, ngrams + 1):
            all_grams.extend(' '.join(g) for g in zip(*[unigrams[i:] for i in range(n)]))
        return all_grams
    return unigrams


# Show preprocessing on a short text
sample = 'Prophet handles holiday effects automatically for seasonal retail demand.'
tokens = chunk_prep(sample)
print(f'Input:  "{sample}"')
print(f'Output: {tokens}')
print(f'Tokens: {len(tokens)} ({len([t for t in tokens if " " not in t])} unigrams, {len([t for t in tokens if " " in t])} bigrams)')

In [ ]:
# Show preprocessing on a chunk from each type
for label, text in [('Reddit', reddit_row.text_content[:200]),
                     ('Zoom', zoom_row.text_content[:200]),
                     ('PDF', pdf_row.text_content[:200])]:
    tokens = chunk_prep(text)
    unigrams = [t for t in tokens if ' ' not in t]
    bigrams = [t for t in tokens if ' ' in t]
    print(f'{label}: {len(tokens)} tokens ({len(unigrams)} unigrams, {len(bigrams)} bigrams)')
    print(f'  First 10 unigrams: {unigrams[:10]}')
    print(f'  First 5 bigrams:   {bigrams[:5]}')
    print()

---
## BM25 Input Construction (Approach B)

We chose **Approach B: text + enrichment tags** for BM25 input construction. The input for each chunk is:

```
text_content + " " + " ".join(topics) + " " + " ".join(type_specific_tags)
```

**Why?** Enrichment tags normalize concepts across source types. A topic like "demand forecasting" tagged on both a Reddit chunk and a PDF chunk produces shared BM25 terms that wouldn't appear in the raw text of both. Without tags, a Reddit chunk discussing Prophet and a PDF chunk documenting `ML.FORECAST` would have minimal vocabulary overlap despite being about the same concept.

The alternative (Approach A: text only) would miss this cross-type normalization. For your use case, you may prefer Approach A if your enrichment tags are noisy or if you want BM25 to reflect only the original text.

In [ ]:
def build_bm25_text(row, source_type):
    """Construct BM25 input: text_content + enrichment tags (Approach B)."""
    parts = [row.text_content]
    if row.topics:
        parts.extend(row.topics)
    # Type-specific tags
    if source_type == 'reddit' and hasattr(row, 'methods_mentioned') and row.methods_mentioned:
        parts.extend(row.methods_mentioned)
    elif source_type == 'zoom' and hasattr(row, 'action_items') and row.action_items:
        parts.extend(row.action_items)
    elif source_type == 'pdf' and hasattr(row, 'functions_referenced') and row.functions_referenced:
        parts.extend(row.functions_referenced)
    return ' '.join(parts)


# Show BM25 input for each type
for label, row, stype in [('Reddit', reddit_row, 'reddit'),
                           ('Zoom', zoom_row, 'zoom'),
                           ('PDF', pdf_row, 'pdf')]:
    bm25_text = build_bm25_text(row, stype)
    print(f'{label} BM25 input ({len(bm25_text)} chars):')
    print(f'  text_content: {len(row.text_content)} chars')
    print(f'  + tags: {len(bm25_text) - len(row.text_content)} chars added')
    print(f'  Preview: {bm25_text[:150]}...')
    print()

---
## Train BM25 Model

Train `BM25Okapi` on the 3 sample chunks. In the full pipeline, `build_bm25.py` (step 7a) trains on all chunks and saves the model to BigQuery, then `apply_bm25.py` (step 7b) computes sparse embeddings and updates BQ + VS2.

In [ ]:
# Build corpus
corpus_texts = [
    build_bm25_text(reddit_row, 'reddit'),
    build_bm25_text(zoom_row, 'zoom'),
    build_bm25_text(pdf_row, 'pdf'),
]
tokenized_corpus = [chunk_prep(text) for text in corpus_texts]

# Train
bm25_model = BM25Okapi(
    tokenized_corpus,
    k1=BM25_K1,
    b=BM25_B,
    epsilon=BM25_EPSILON,
)

# Build vocabulary mapping
vocabulary = bm25_model.idf.keys()
vocab_map = {word: i for i, word in enumerate(vocabulary)}

print(f'Vocabulary size: {len(vocabulary)}')
print(f'Average document length (tokens): {bm25_model.avgdl:.1f}')
print(f'Corpus size: {len(tokenized_corpus)} documents')
print(f'\nDocument lengths: {[len(t) for t in tokenized_corpus]}')

# Top IDF terms (most discriminating)
idf_sorted = sorted(bm25_model.idf.items(), key=lambda x: x[1], reverse=True)
print(f'\nTop 15 IDF terms (most discriminating):')
for term, idf in idf_sorted[:15]:
    print(f'  {idf:.4f}  {term}')

---
## Compute Sparse Embedding

Convert a chunk's BM25 scores into a sparse vector — a list of `(index, value)` pairs where `index` maps to a vocabulary term and `value` is the BM25 score. Only non-zero terms are stored.

In [ ]:
def create_bm25_sparse_embedding(text, bm25_model, vocab_mapping):
    """Compute BM25 sparse embedding for text using trained model."""
    tokens = chunk_prep(text)
    term_freqs = Counter(tokens)
    doc_len = len(tokens)
    k1, b, avgdl = bm25_model.k1, bm25_model.b, bm25_model.avgdl

    indices, values = [], []
    for term, freq in term_freqs.items():
        if term not in vocab_mapping:
            continue
        idf = bm25_model.idf[term]
        score = idf * (freq * (k1 + 1)) / (freq + k1 * (1 - b + b * doc_len / avgdl))
        indices.append(vocab_mapping[term])
        values.append(score)
    return indices, values


# Compute for the Reddit chunk
reddit_bm25_text = build_bm25_text(reddit_row, 'reddit')
indices, values = create_bm25_sparse_embedding(reddit_bm25_text, bm25_model, vocab_map)

print(f'Reddit chunk: {reddit_row.chunk_id}')
print(f'  Total vocabulary: {len(vocab_map)}')
print(f'  Non-zero terms: {len(indices)}')
print(f'  Sparsity: {1 - len(indices) / len(vocab_map):.1%}')
print(f'  Score range: [{min(values):.4f}, {max(values):.4f}]')

# Show top-scoring terms
reverse_vocab = {i: term for term, i in vocab_map.items()}
scored = sorted(zip(indices, values), key=lambda x: x[1], reverse=True)
print(f'\n  Top 10 scoring terms:')
for idx, val in scored[:10]:
    print(f'    {val:.4f}  {reverse_vocab[idx]}')

---
## Query Embedding

The same `create_bm25_sparse_embedding` function is used at query time. The query text is preprocessed identically — this ensures vocabulary alignment between indexed documents and search queries.

In [ ]:
query = 'What machine learning methods work best for demand forecasting?'

q_indices, q_values = create_bm25_sparse_embedding(query, bm25_model, vocab_map)

print(f'Query: "{query}"')
print(f'  Non-zero terms: {len(q_indices)}')

q_scored = sorted(zip(q_indices, q_values), key=lambda x: x[1], reverse=True)
print(f'\n  Query terms and scores:')
for idx, val in q_scored:
    print(f'    {val:.4f}  {reverse_vocab[idx]}')

# Check overlap with each chunk
q_index_set = set(q_indices)
for label, row, stype in [('Reddit', reddit_row, 'reddit'),
                           ('Zoom', zoom_row, 'zoom'),
                           ('PDF', pdf_row, 'pdf')]:
    bm25_text = build_bm25_text(row, stype)
    c_indices, c_values = create_bm25_sparse_embedding(bm25_text, bm25_model, vocab_map)
    overlap = q_index_set & set(c_indices)
    overlap_terms = [reverse_vocab[i] for i in overlap]
    print(f'\n  Overlap with {label}: {len(overlap)}/{len(q_indices)} query terms')
    print(f'    Matching terms: {overlap_terms}')